# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [ ]:
# Task — Multi-series line with highlight
asia = df[df['Region'] == 'Asia'].sort_values(['Country', 'Year'])
highlight = 'China'

start_yr, end_yr = asia['Year'].min(), asia['Year'].max()
hl = asia[asia['Country'] == highlight]
hl_start = hl.loc[hl['Year'] == start_yr, 'CO2_Mt'].iloc[0]
hl_end = hl.loc[hl['Year'] == end_yr, 'CO2_Mt'].iloc[0]
growth = hl_end / hl_start

fig = go.Figure()

# Context: all other Asian countries in light grey, thin, labelled at line end
for country, g in asia.groupby('Country'):
    if country == highlight:
        continue
    fig.add_trace(go.Scatter(
        x=g['Year'], y=g['CO2_Mt'], mode='lines',
        line=dict(color='#EEEEEE', width=1.5),
        hovertemplate=f'{country}: %{{y:,.0f}} Mt<extra></extra>'
    ))
    fig.add_annotation(
        x=g['Year'].iloc[-1], y=g['CO2_Mt'].iloc[-1], text=country,
        xanchor='left', xshift=6, showarrow=False,
        font=dict(size=11, color='#AAAAAA')
    )

# Highlight: drawn last so it sits on top
fig.add_trace(go.Scatter(
    x=hl['Year'], y=hl['CO2_Mt'], mode='lines',
    line=dict(color='#C0392B', width=3.5),
    hovertemplate=f'{highlight}: %{{y:,.0f}} Mt<extra></extra>'
))
fig.add_annotation(
    x=end_yr, y=hl_end, text=f'<b>{highlight}</b><br>{hl_end:,.0f} Mt',
    xanchor='left', xshift=6, showarrow=False, align='left',
    font=dict(size=13, color='#C0392B')
)

fig.update_layout(
    title=dict(
        text=(f"<b>{highlight}'s CO2 emissions grew {growth:.1f}x since {start_yr}, "
              f"far outpacing the rest of Asia</b>"
              f"<br><sup>Annual CO2 emissions (million tonnes), Asian countries, {start_yr}–{end_yr}</sup>"),
        x=0, xanchor='left'
    ),
    template='simple_white',
    font=dict(family='Arial', size=13),
    showlegend=False,
    xaxis=dict(title=None, showline=False, ticks='', tickangle=0, dtick=5),
    yaxis=dict(title='CO2 emissions (Mt)', showline=False, ticks='',
               showgrid=True, gridcolor='#F2F2F2', rangemode='tozero',
               tickformat=','),
    height=520, margin=dict(l=20, r=120, t=100, b=40)
)
fig.show()
